# 12 - Matriz de experimentos con los hiperparámetros ajustados
En este notebook comparamos la matriz completa de experimentos del notebook 10 con la misma matriz usando los hiperparámetros ajustados en el notebook 11.

Las dos matrices se han ejecutado con `make experiments` (`urban_accessibility_ml/modeling/experiments_run.py`), con las mismas particiones, conjuntos de variables, perfiles y modelos: una con los valores por defecto de cada modelo y otra con los hiperparámetros ajustados de cada partición y conjunto de variables. Tardan varios minutos, así que aquí solo se leen los resultados, que están en `reports/results/`:
- `experiment-matrix-default.csv`: con los valores por defecto.
- `experiment-matrix-tuned.csv`: con los hiperparámetros ajustados.

El ajuste solo ha usado los datos de entrenamiento de cada partición, así que el test sigue sin haberse usado para elegir nada. Aquí vemos cuánto mejoran los modelos en el test.

## Carga de los resultados

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from urban_accessibility_ml.config import REPORTS_DIR
from urban_accessibility_ml.modeling.evaluate import summarize_folds
from urban_accessibility_ml.plotting.palettes import SERIE_BASE
from urban_accessibility_ml.plotting.plots import SERIES, use_style

DEFAULT_FILE = REPORTS_DIR / "results" / "experiment-matrix-default.csv"
TUNED_FILE = REPORTS_DIR / "results" / "experiment-matrix-tuned.csv"
GROUP = ["strategy", "model"]
STRATEGIES = ["within_city", "mixed", "cross_city"]

use_style()

results_default = pd.read_csv(DEFAULT_FILE)
results_tuned = pd.read_csv(TUNED_FILE)
len(results_default), len(results_tuned)

## Comparación por estrategia y modelo
F1 macro media en el test, con los valores por defecto y con los ajustados. Cada fila contiene los promedios de todos los conjuntos de variables, perfiles y pliegues de esa estrategia y modelo. La última columna es la diferencia: si es positiva, el ajuste mejora el resultado.

In [ ]:
default = summarize_folds(results_default, GROUP).set_index(GROUP)["f1_macro_mean"]
tuned = summarize_folds(results_tuned, GROUP).set_index(GROUP)["f1_macro_mean"]

comparison = pd.DataFrame({"por defecto": default, "ajustado": tuned})
comparison["diferencia"] = comparison["ajustado"] - comparison["por defecto"]
comparison.round(3)

In [ ]:
fig, axes = plt.subplots(1, len(STRATEGIES), figsize=(13, 4), sharey=True)
width = 0.38

for ax, strategy in zip(axes, STRATEGIES):
    part = comparison.loc[strategy]
    x = np.arange(len(part))
    ax.bar(x - width / 2, part["por defecto"], width, color=SERIE_BASE, label="por defecto")
    ax.bar(x + width / 2, part["ajustado"], width, color=SERIES[0], label="ajustado")
    ax.set_xticks(x, part.index, rotation=30, ha="right")
    ax.set_title(strategy)

axes[0].set_ylabel("F1 macro (test)")
axes[0].legend()
plt.tight_layout()
plt.show()

Vemos que en la regresión logística sí se aprecian mejoras, sobre todo en misma ciudad (pasa de 0,754 a 0,831) y en mixto (de 0,747 a 0,812). En cambio, en cross_city casi no cambia.

En los modelos de árboles la historia es distinta: no se aprecian mejoras y, en algunos casos, incluso empeoran un poco. Las diferencias son pequeñas, y la mayor caída es la de XGBoost en cross_city (de 0,446 a 0,429).

## Comparación por conjunto de variables
Mostramos la diferencia de F1 macro media en el test para cada modelo y conjunto de variables, con todas las estrategias, perfiles y pliegues. Si es positiva, el ajuste mejora el resultado.

In [ ]:
BY_SET = ["feature_set", "model"]
default_set = summarize_folds(results_default, BY_SET).set_index(BY_SET)["f1_macro_mean"]
tuned_set = summarize_folds(results_tuned, BY_SET).set_index(BY_SET)["f1_macro_mean"]
(tuned_set - default_set).unstack("model").drop(columns="dummy").round(3)

## Otras métricas
Los hiperparámetros se han elegido con la F1 macro, así que es la métrica en la que el ajuste parte con ventaja. Para contrastarlo, mostramos la misma diferencia (ajustado menos por defecto) con la balanced accuracy y el kappa cuadrático, primero por estrategia y modelo, y después por conjunto de variables y modelo.

In [ ]:
EXTRA_METRICS = ["balanced_accuracy", "quadratic_weighted_kappa"]


def difference(metric, group):
    """Tuned minus default mean of a metric, one column per model (without the dummy)."""
    default = summarize_folds(results_default, group).set_index(group)[f"{metric}_mean"]
    tuned = summarize_folds(results_tuned, group).set_index(group)[f"{metric}_mean"]
    return (tuned - default).unstack("model").drop(columns="dummy")


pd.concat({m: difference(m, ["strategy", "model"]) for m in EXTRA_METRICS}, names=["métrica"]).round(3)

In [ ]:
pd.concat({m: difference(m, ["feature_set", "model"]) for m in EXTRA_METRICS}, names=["métrica"]).round(3)

In [ ]:
import seaborn as sns

from urban_accessibility_ml.config import FEATURE_SETS
from urban_accessibility_ml.plotting.plots import CMAP_DIV, save_figure

ALL_METRICS = {
    "f1_macro": "F1 macro",
    "balanced_accuracy": "Balanced accuracy",
    "quadratic_weighted_kappa": "Kappa cuadrático",
}
LIMIT = 0.12

fig, axes = plt.subplots(1, len(ALL_METRICS), figsize=(13, 3.6), sharey=True)
for ax, (metric, title) in zip(axes, ALL_METRICS.items()):
    sns.heatmap(
        difference(metric, ["feature_set", "model"]).reindex(FEATURE_SETS),
        annot=True, fmt="+.3f", cmap=CMAP_DIV, center=0, vmin=-LIMIT, vmax=LIMIT,
        cbar=False, linewidths=1.5, linecolor="white", ax=ax, annot_kws={"size": 8},
    )
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(axis="x", rotation=30)
    ax.tick_params(axis="y", rotation=0)
plt.tight_layout()
save_figure("diferencia_por_conjunto_y_metrica", "0.12-matriz-ajustada", fig=fig)
plt.show()

Con la balanced accuracy y la kappa ponderado cuadrática vemos que la mejora de la regresión logística es diferente según el conjunto de variables y de la métrica.
- En `completo` es clara en F1 macro (+0,115) y en kappa cuatrática (+0,120), y más modesta en balanced accuracy (+0,032).
- En `calle` y `combinado` la mejora de F1 macro (+0,062) se reduce mucho en las otras dos métricas (+0,005 y +0,023)
- Y en `cartografico_aereo` hay mejora muy moderada (+0,019 en F1 macro y +0,005 en kappa) e incluso baja la balanced accuracy (−0,017).

En los modelos de árboles no hay mejoras apreciables en ninguna de las tres métricas. Las pocas diferencias positivas son:
- Un +0,004 como máximo (por ejemplo, random forest en `calle` y `combinado` en balanced accuracy).
- La mayor caída por conjunto de variables es la de XGBoost en `cartografico_aereo` (−0,027 en F1 macro y −0,038 en kappa).